# Step 2 – Train CNN for Line Following

Fine-tunes a **ResNet18** (pretrained on ImageNet) to classify each camera frame as
`forward`, `left`, or `right`. The trained model is saved as `line_follower.pth`.

**Prerequisites:** Run Step 1 first and collect at least 200 images per class.

In [ ]:
# ── Cell 1: Check dataset ─────────────────────────────────────────────────────
import os
classes = ['forward', 'left', 'right']
for cls in classes:
    path = f'dataset/{cls}'
    n = len(os.listdir(path)) if os.path.isdir(path) else 0
    print(f'  {cls}: {n} images')
    if n < 50:
        print(f'  WARNING: {cls} has very few images — collect more for better accuracy.')

In [ ]:
# ── Cell 2: Training ──────────────────────────────────────────────────────────
import torch
import torchvision
import torchvision.transforms as transforms
from torchvision.datasets import ImageFolder
from torch.utils.data import DataLoader, random_split
import torch.nn as nn
import torch.optim as optim

# ── Hyper-parameters ──────────────────────────────────────────────────────────
BATCH_SIZE  = 32
EPOCHS      = 15
LR          = 1e-3
VAL_SPLIT   = 0.2    # 20 % held out for validation
SAVE_PATH   = 'line_follower.pth'

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Training on:', device)

# ── Data augmentation (training) and normalisation ────────────────────────────
# Images were saved as 224×112 in Step 1; resize to 224×224 for ResNet
train_tf = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),          # augment: mirror scene
    transforms.ColorJitter(brightness=0.3,
                           contrast=0.3,
                           saturation=0.2),     # augment: lighting variation
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std =[0.229, 0.224, 0.225]),
])
val_tf = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std =[0.229, 0.224, 0.225]),
])

# Load full dataset then split
full_dataset = ImageFolder('dataset', transform=train_tf)
class_names  = full_dataset.classes          # ['forward', 'left', 'right'] — alphabetical
print('Classes (index order):', class_names)

n_val   = int(len(full_dataset) * VAL_SPLIT)
n_train = len(full_dataset) - n_val
train_ds, val_ds = random_split(full_dataset, [n_train, n_val],
                                generator=torch.Generator().manual_seed(42))
# Apply val transform to val subset
val_ds.dataset = ImageFolder('dataset', transform=val_tf)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,  num_workers=2)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False, num_workers=2)
print(f'Train: {n_train}  Val: {n_val}')

# ── Model: ResNet18 fine-tuned ────────────────────────────────────────────────
model = torchvision.models.resnet18(weights='IMAGENET1K_V1')
# Replace final FC layer: 512 → 3 classes
model.fc = nn.Linear(model.fc.in_features, len(class_names))
model = model.to(device)

criterion = nn.CrossEntropyLoss()
# Use different LRs: lower for backbone, higher for new head
optimizer = optim.Adam([
    {'params': [p for n, p in model.named_parameters() if 'fc' not in n], 'lr': LR * 0.1},
    {'params': model.fc.parameters(), 'lr': LR}
])
scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=5, gamma=0.5)

# ── Training loop ─────────────────────────────────────────────────────────────
best_val_acc = 0.0

for epoch in range(1, EPOCHS + 1):
    # --- Train ---
    model.train()
    running_loss, correct, total = 0.0, 0, 0
    for imgs, labels in train_loader:
        imgs, labels = imgs.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(imgs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item() * imgs.size(0)
        correct += (outputs.argmax(1) == labels).sum().item()
        total   += imgs.size(0)

    train_loss = running_loss / total
    train_acc  = correct / total

    # --- Validate ---
    model.eval()
    v_correct, v_total = 0, 0
    with torch.no_grad():
        for imgs, labels in val_loader:
            imgs, labels = imgs.to(device), labels.to(device)
            outputs = model(imgs)
            v_correct += (outputs.argmax(1) == labels).sum().item()
            v_total   += imgs.size(0)
    val_acc = v_correct / v_total

    scheduler.step()

    print(f'Epoch {epoch:2d}/{EPOCHS}  '
          f'loss: {train_loss:.4f}  train_acc: {train_acc:.3f}  val_acc: {val_acc:.3f}')

    # Save the best checkpoint
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save({
            'model_state_dict': model.state_dict(),
            'class_names':      class_names,
            'val_acc':          val_acc,
        }, SAVE_PATH)
        print(f'  → Saved new best model (val_acc={val_acc:.3f})')

print(f'\nTraining complete. Best val accuracy: {best_val_acc:.3f}')
print(f'Model saved to: {SAVE_PATH}')

In [ ]:
# ── Cell 3: Quick confusion matrix ───────────────────────────────────────────
# Shows how well the model distinguishes each steering class.
import numpy as np

# Load best weights
ckpt = torch.load(SAVE_PATH, map_location=device)
model.load_state_dict(ckpt['model_state_dict'])
model.eval()

all_preds, all_labels = [], []
with torch.no_grad():
    for imgs, labels in val_loader:
        preds = model(imgs.to(device)).argmax(1).cpu().numpy()
        all_preds.extend(preds)
        all_labels.extend(labels.numpy())

all_preds  = np.array(all_preds)
all_labels = np.array(all_labels)

print('Confusion matrix (rows=true, cols=predicted):')
print(f'         {", ".join(f"{c:>8}" for c in class_names)}')
for i, cls in enumerate(class_names):
    row = [(all_preds[all_labels == i] == j).sum() for j in range(len(class_names))]
    print(f'{cls:>8}  {", ".join(f"{v:8d}" for v in row)}')
print(f'\nOverall val accuracy: {(all_preds == all_labels).mean():.3f}')